# MVP2 FDTD → HDMI bring-up

Real-time 2D FDTD electromagnetic wave, rendered as a ray-marched 3D heightmap on **HDMI** (640×480).

**What you should see:** plug HDMI into the PYNQ-Z1 J11 port → a shaded 3D terrain whose hills are the live |E| (or |S|) wave radiating from the injected source, updating continuously.

Pipeline (all on the 25 MHz pixel clock): `CORDIC → FDTD solver → field magnitude → ping-pong s_mag → bridge → 52 heightmap BRAMs → ray_unit → rgb2dvi → HDMI`.

## AXI map
| Base | Block | Use |
|------|-------|-----|
| `0x4000_0000` | renderer camera | optional camera pose (defaults are fine) |
| `0x4120_0000` | gpio_ctrl | FDTD control |
| `0x4121_0000` | gpio_status | FDTD status / checksum |

`gpio_ctrl` CH1 = `{amplitude[31:16], phase_step[15:0]}`, CH2 = `{free_run[15], sample_req[14], mag_mode[13], solver_enable[12], source_addr[11:0]}`.

`gpio_status` CH1 = `checksum[31:0]`, CH2 = `{source_q313[31:16], bridge_busy[7], pp_frame_ready[6], pp_read_sel[5], source_latched[4], mag_busy[3], mag_done[2], source_valid[1], solver_done[0]}`.

In [ ]:
from pynq import Overlay, MMIO
import time

# Copy fdtd_hdmi.bit (renamed from fdtd_hdmi_bd_wrapper.bit) and the matching .hwh
# to the board with the SAME basename, e.g. fdtd_hdmi.bit / fdtd_hdmi.hwh
ol = Overlay('fdtd_hdmi.bit')
print('Overlay loaded')
print([k for k in ol.ip_dict.keys()])

In [ ]:
CTRL   = MMIO(0x41200000, 0x10000)
STATUS = MMIO(0x41210000, 0x10000)
CAMERA = MMIO(0x40000000, 0x1000)

GPIO_CH1 = 0x0   # data register, channel 1
GPIO_CH2 = 0x8   # data register, channel 2

GRID = 64
def cell(x, y):
    return y * GRID + x

def q313(v):
    """float -> signed Q3.13 16-bit (1.0 = 8192)"""
    iv = int(round(v * 8192)) & 0xFFFF
    return iv

def set_ctrl(phase_step, amplitude, source_addr, solver_enable, mag_mode, sample_req, free_run):
    ch1 = (q313(amplitude) << 16) | q313(phase_step)
    ch2 = ((free_run & 1) << 15) | ((sample_req & 1) << 14) | ((mag_mode & 1) << 13) \
        | ((solver_enable & 1) << 12) | (source_addr & 0xFFF)
    CTRL.write(GPIO_CH1, ch1)
    CTRL.write(GPIO_CH2, ch2)

def read_status():
    chk = STATUS.read(GPIO_CH1)
    s   = STATUS.read(GPIO_CH2)
    return {
        'checksum'      : chk,
        'solver_done'   : (s >> 0) & 1,
        'source_valid'  : (s >> 1) & 1,
        'mag_done'      : (s >> 2) & 1,
        'mag_busy'      : (s >> 3) & 1,
        'source_latched': (s >> 4) & 1,
        'pp_read_sel'   : (s >> 5) & 1,
        'pp_frame_ready': (s >> 6) & 1,
        'bridge_busy'   : (s >> 7) & 1,
        'source_q313'   : (s >> 16) & 0xFFFF,
    }
print('helpers ready')

## 1. Start the simulation in free-run mode

Inject a sine source at the grid centre, enable the solver in **free-run** (auto-restart after every magnitude pass), and hold `sample_req` high so the CORDIC keeps advancing the source phase.

In [ ]:
set_ctrl(
    phase_step   = 0.05,            # source angular step per CORDIC sample (frequency)
    amplitude    = 0.9,            # source amplitude (Q3.13, < 1.0)
    source_addr  = cell(32, 32),  # centre of the 64x64 grid
    solver_enable= 1,
    mag_mode     = 0,             # 0 = |E|, 1 = |S| (Poynting)
    sample_req   = 1,
    free_run     = 1,
)
print('FDTD running. source_addr =', cell(32,32))

## 2. Confirm the solver is alive

The checksum must keep changing (solver writing fields), `pp_frame_ready`/`pp_read_sel` should toggle as frames complete, and `bridge_busy` pulses once per display vblank as it copies the wave into the heightmap.

In [ ]:
seen = set()
read_sel_flips = 0
prev_sel = None
for i in range(20):
    st = read_status()
    seen.add(st['checksum'])
    if prev_sel is not None and st['pp_read_sel'] != prev_sel:
        read_sel_flips += 1
    prev_sel = st['pp_read_sel']
    print(f"chk=0x{st['checksum']:08x} done={st['solver_done']} mag_done={st['mag_done']} "
          f"read_sel={st['pp_read_sel']} frame_rdy={st['pp_frame_ready']} bridge={st['bridge_busy']} "
          f"src=0x{st['source_q313']:04x}")
    time.sleep(0.05)

print()
print('unique checksums seen :', len(seen), '(should be > 1 => solver active)')
print('read_sel flips        :', read_sel_flips, '(> 0 => ping-pong swapping)')
assert len(seen) > 1, 'checksum frozen — solver not running'
print('PASS: solver + ping-pong are live.')

## 3. Look at the HDMI output

There is no framebuffer to read from the PS — the renderer streams straight to HDMI. **Check the monitor**: you should see a 3D terrain with concentric ripples expanding from the centre, damped at the edges by the PML boundary.

Try tuning the source live (re-run with different values):
* `phase_step` — wave frequency.
* `amplitude` — wave height.
* `mag_mode=1` — switch to |S| (Poynting) magnitude.
* `source_addr = cell(20, 40)` — move the source off-centre.

## 4. (Optional) camera pose

The renderer boots with a fixed default camera, so HDMI works without touching `0x4000_0000`. The `camera_ctrl_axi` block accepts a new origin/basis written to its shadow registers followed by a commit; it is applied at the next frame boundary (vsync) for tear-free camera moves. See `rtl/renderer/camera_ctrl_axi.sv` for the exact register offsets if you want to orbit the camera from the PS.